[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab08_llm_prompt_engineering.ipynb)

# ปฏิบัติการที่ 8: แบบจำลองภาษาขนาดใหญ่และการออกแบบพรอมต์ (Prompt Engineering) สำหรับข้อความทางคลินิก

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันเสาร์ที่ 17 ตุลาคม 2026 (ภาคบ่าย) · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- สามารถรันแบบจำลองภาษาขนาดใหญ่ (large language model, LLM) แบบเปิดบน Colab โดยไม่ต้องใช้ API key และไม่มีค่าใช้จ่าย
- สามารถออกแบบพรอมต์ (prompt) สำหรับงานทางคลินิก ได้แก่ การสรุปความ (summarization) การสกัดข้อมูล (information extraction) และการถาม-ตอบ (question answering)
- สามารถกำหนดให้แบบจำลองตอบเป็นข้อมูลมีโครงสร้าง (JSON) เพื่อนำไปใช้งานต่อ
- ตระหนักถึงการสร้างข้อมูลเท็จ (hallucination) และข้อจำกัดด้านความปลอดภัยของ LLM ในงานสุขภาพ

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (กด Shift+Enter หรือคลิกปุ่ม ▶) ในปฏิบัติการนี้ ผู้เรียนเพียงแก้ไขข้อความพรอมต์ภายในเครื่องหมายอัญประกาศ `"..."` แล้วรันเซลล์ใหม่ ส่วนโค้ดอื่นได้จัดเตรียมไว้แล้ว แบบฝึกหัดทุกข้อมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ให้ผู้เรียนทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ ซึ่งเป็นชิ้นงานที่ต้องส่ง หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom
>
> 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง

> ⚡ **จำเป็นต้องใช้ GPU:** ก่อนเริ่มปฏิบัติการ ให้เลือก `Runtime → Change runtime type → T4 GPU → Save` (ไม่มีค่าใช้จ่าย) หากไม่ใช้ GPU แบบจำลองยังทำงานได้ แต่แต่ละคำตอบจะใช้เวลาประมาณ 1 นาที

> 🔒 **จริยธรรม:** ห้ามนำข้อมูลผู้ป่วยจริงไปป้อนให้แบบจำลองปัญญาประดิษฐ์ (AI) หรือเว็บไซต์ใด ๆ บันทึกทางคลินิกทั้งหมดในปฏิบัติการนี้เป็นข้อมูลสังเคราะห์

> 🌐 **ภาษาที่ใช้:** บันทึกทางคลินิกและพรอมต์ที่ส่งให้แบบจำลองในปฏิบัติการนี้เป็นภาษาอังกฤษ เนื่องจากแบบจำลองขนาดเล็กทำงานได้ดีที่สุดกับภาษาอังกฤษ และชุดข้อมูล (dataset) เป็นภาษาอังกฤษ ทุกเซลล์มีคำอธิบายภาษาไทยกำกับความหมายของพรอมต์ และมีเซลล์สำหรับทดลองสั่งงานเป็นภาษาไทยเพื่อเปรียบเทียบ

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab08"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## 1. การโหลดแบบจำลองภาษาขนาดใหญ่แบบเปิดบน Colab

**แบบจำลองภาษาขนาดใหญ่ (LLM)** คือโปรแกรมที่ผ่านการฝึกจากข้อความจำนวนมากเพื่อทำนายคำถัดไป ส่วน LLM ชนิด *instruction-tuned* ได้รับการฝึกเพิ่มเติมให้ปฏิบัติตามคำสั่งในลักษณะผู้ช่วยสนทนา ปฏิบัติการนี้ใช้แบบจำลอง **แบบเปิด (open model)** ขนาดเล็กจาก Hugging Face ชื่อ `Qwen2.5-1.5B-Instruct` (1.5 พันล้านพารามิเตอร์) ซึ่งทำงานบน GPU T4 ของ Colab รุ่นไม่มีค่าใช้จ่ายได้ โดยไม่ต้องใช้ API key และข้อความไม่ถูกส่งออกนอกเซสชัน Colab ของผู้เรียน หลักการออกแบบพรอมต์ในปฏิบัติการนี้สามารถนำไปใช้กับแบบจำลองขนาดใหญ่กว่าได้เช่นกัน (ChatGPT, Claude, Gemini เป็นต้น)

ให้รันเซลล์ถัดไป การรันครั้งแรกจะดาวน์โหลดแบบจำลอง (ประมาณ 3 GB) และใช้เวลาประมาณ **2–4 นาที**

In [ ]:
# @title 🤖 โหลดแบบจำลองและฟังก์ชันช่วย `chat()` — ให้รันเซลล์นี้โดยไม่ต้องแก้ไข
!pip -q install "transformers>=4.56" accelerate
import torch, transformers
from transformers import pipeline
transformers.logging.set_verbosity_error()   # ซ่อนคำเตือนทางเทคนิค

if torch.cuda.is_available():
    print("✅ พบ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ ไม่พบ GPU — โปรดเลือก Runtime → Change runtime type → T4 GPU → Save แล้วรันทุกเซลล์ใหม่\n"
          "   (สามารถทำต่อบน CPU ได้ แต่แต่ละคำตอบจะใช้เวลาประมาณ 1 นาที)")

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
llm = pipeline("text-generation", model=MODEL,
               dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

def chat(system, user, max_new_tokens=300, examples=None):
    """ส่งคำขอ 1 ครั้งไปยังแบบจำลอง และคืนค่าคำตอบเป็นข้อความ
    system = บทบาทและกฎของแบบจำลอง;  user = คำขอ;  examples = (ไม่บังคับ) รายการตัวอย่างคู่ (คำขอ, คำตอบ)"""
    messages = [{"role": "system", "content": system}]
    for ex_user, ex_reply in (examples or []):
        messages += [{"role": "user", "content": ex_user}, {"role": "assistant", "content": ex_reply}]
    messages.append({"role": "user", "content": user})
    out = llm(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

print(chat("You are a helpful medical assistant.", "In one sentence, what is hypertension?"))

💡 **คำอธิบายผลลัพธ์**
เซลล์ข้างต้นดาวน์โหลดแบบจำลองและจัดเตรียมฟังก์ชันช่วย `chat(system, user)` ข้อความ **system** กำหนดบทบาทและกฎของแบบจำลอง ส่วนข้อความ **user** คือคำขอ (ตัวอย่างข้างต้นขอให้อธิบาย hypertension ใน 1 ประโยค) พารามิเตอร์ `max_new_tokens` จำกัดความยาวของคำตอบ (*token* คือคำหรือส่วนของคำ) และ `do_sample=False` ทำให้พรอมต์เดิมให้คำตอบเดิมทุกครั้ง จึงสามารถเปรียบเทียบพรอมต์ต่าง ๆ ได้อย่างเป็นธรรม

In [ ]:
# @title 🧰 ฟังก์ชันช่วยสำหรับจัดการคำตอบแบบ JSON — ให้รันเซลล์นี้โดยไม่ต้องแก้ไข
import json, re

def parse_json_reply(raw, quiet=False):
    """ค้นหาบล็อก {...} แรกในคำตอบของแบบจำลอง (ข้ามเครื่องหมาย ```json และข้อความส่วนเกิน)
    และแปลงเป็น dictionary ของ Python คืนค่า {} หากไม่พบ JSON ที่ถูกต้อง"""
    text = re.sub(r"```(?:json)?", "", str(raw))
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text[m.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    if not quiet:
        print("⚠️ ไม่พบ JSON ที่ถูกต้องในคำตอบ — โปรดปรับพรอมต์ให้กำหนดรูปแบบอย่างเข้มงวดขึ้น")
    return {}

def _as_list(v):
    if v is None:
        return []
    if isinstance(v, str):
        return [s.strip() for s in re.split(r"[;,]", v) if s.strip()]
    if isinstance(v, dict):
        v = list(v.values())
    if not isinstance(v, list):
        return [str(v)]
    return [str(x.get("name", x.get("drug", "")) if isinstance(x, dict) else x).strip() for x in v if x]

def clean_record(d):
    """คงไว้เฉพาะ age / sex / conditions / medications ในรูปแบบที่ตารางอันดับกำหนด"""
    d = d if isinstance(d, dict) else {}
    age = None
    m = re.search(r"\d+", str(d.get("age", "")))
    if m:
        age = int(m.group(0))
    sex = str(d.get("sex", "")).strip().lower()
    sex = {"m": "M", "male": "M", "man": "M", "f": "F", "female": "F", "woman": "F"}.get(sex)
    return {"age": age, "sex": sex,
            "conditions": _as_list(d.get("conditions")), "medications": _as_list(d.get("medications"))}

def show_record(data, required=("age", "sex", "conditions", "medications")):
    """แสดงข้อมูลที่สกัดได้ในรูปแบบที่อ่านง่าย และระบุคีย์ที่ขาดหายไป"""
    if data is ... or not isinstance(data, dict):
        print("⚠️ `data` ยังไม่เป็น dictionary — โปรดตรวจสอบว่าได้แทนที่ ... แล้ว")
        return
    print(json.dumps(data, indent=2, ensure_ascii=False))
    missing = [k for k in required if k not in data]
    print("\n✅ มีคีย์ที่จำเป็นครบถ้วน" if not missing else f"\n⚠️ คีย์ที่ขาดหายไป: {missing}")

print("ฟังก์ชันช่วยพร้อมใช้งาน: parse_json_reply(), clean_record(), show_record()")

## 2. องค์ประกอบของพรอมต์ (prompt patterns)

พรอมต์ทางคลินิกที่ดีควรระบุองค์ประกอบ 4 ประการ ดังนี้

| องค์ประกอบ | ตัวอย่าง (ภาษาอังกฤษที่ใช้ในปฏิบัติการ) |
|---|---|
| **บทบาท (role)** — แบบจำลองทำหน้าที่ใด | "You are a clinical documentation assistant." (ผู้ช่วยจัดทำเวชระเบียน) |
| **งาน (task)** — ให้ดำเนินการสิ่งใด | "Summarize this note." (สรุปบันทึกนี้) |
| **ข้อจำกัด (constraints)** — กฎและขอบเขต | "Use only facts in the note. Do not guess." (ใช้เฉพาะข้อมูลในบันทึก ห้ามคาดเดา) |
| **รูปแบบผลลัพธ์ (output format)** — ลักษณะของคำตอบ | "Exactly 3 bullet points." / "Return JSON." (3 หัวข้อพอดี / ตอบเป็น JSON) |

เซลล์ถัดไปใช้พรอมต์ **ที่ไม่ชัดเจน** (ระบุเพียงบทบาทและงาน) กับบันทึกการติดตามอาการสังเคราะห์ 1 ฉบับ (ผู้ป่วยชายอายุ 57 ปี โรคเบาหวานชนิดที่ 2)

In [ ]:
note = """57M with T2DM presents for follow-up. Reports polyuria and fatigue x2 weeks.
BP 148/92. Fasting glucose 142 mg/dL, HbA1c 8.1%. On metformin 500mg BID.
No chest pain. Plan: increase metformin, add lifestyle counseling, recheck in 6 weeks."""

print("--- พรอมต์ที่ไม่ชัดเจน ---")
print(chat("You are a clinical assistant.", f"Summarize: {note}"))

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: การเขียนพรอมต์สรุปบันทึกให้มีคุณภาพดีขึ้น
# เป้าหมายทางคลินิก: สรุปบันทึกอย่างกระชับและปลอดภัย สำหรับส่งต่อแพทย์ปฐมภูมิ (primary-care physician)
# ✏️ สิ่งที่ต้องทำ: แก้ไขเฉพาะข้อความภายในเครื่องหมายอัญประกาศ (เขียนเป็นภาษาอังกฤษ) แล้วรันเซลล์ (Shift+Enter)
#   1. ใน `system` ให้กำหนดบทบาทของแบบจำลอง
#   2. ใน `user` ให้ระบุจำนวนหัวข้อของสรุป ผู้อ่าน และข้อกำหนดให้ใช้เฉพาะข้อมูลที่ปรากฏในบันทึก
# 💡 คำใบ้: ตัวอย่างเช่น "Write exactly 3 bullet points for a primary-care physician. Use only information
#    in the note; do not add anything."  ห้ามลบ {note} เนื่องจากเป็นตำแหน่งที่บันทึกจะถูกแทรก

system = "You are ✏️ ..."

user = f"""✏️ Write your instructions here.

Note:
{note}"""

if "✏️" in system + user:
    print("⚠️ โปรดแทนที่ส่วนที่มีเครื่องหมาย ✏️ ด้วยข้อความของผู้เรียน (หรือรันเซลล์เฉลยด้านล่าง)\n")
print(chat(system, user))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ความหมายของพรอมต์: บทบาท = ผู้ช่วยจัดทำเวชระเบียนที่ไม่เพิ่มเติมข้อมูลเอง; สรุปสำหรับแพทย์ปฐมภูมิ 3 หัวข้อ
# (ผู้ป่วยและปัญหาหลัก / สิ่งตรวจพบที่สำคัญ / แผนการรักษา) ใช้เฉพาะข้อมูลในบันทึก ความยาวไม่เกิน 60 คำ
system = ("You are an experienced clinical documentation assistant. You write concise, accurate summaries "
          "and never add information that is not in the source note.")
user = f"""Summarize the clinical note below for a primary-care physician.
Rules:
- Exactly 3 bullet points: (1) patient and main problem, (2) key findings, (3) plan.
- Use ONLY information stated in the note. Do not invent values, diagnoses or medications.
- Keep it under 60 words.

Note:
{note}"""
print(chat(system, user))

💡 **คำอธิบายผลลัพธ์**
การแก้ไขเพียงถ้อยคำของพรอมต์ โดยไม่เปลี่ยนโค้ด ทำให้ผลลัพธ์เปลี่ยนไป การระบุ **ผู้อ่าน** **รูปแบบ** และข้อจำกัด **"ใช้เฉพาะข้อมูลในบันทึก"** มักทำให้ได้สรุปที่กระชับ สม่ำเสมอ และปลอดภัยกว่า กระบวนการนี้เรียกว่า *การออกแบบพรอมต์ (prompt engineering)*

🌐 **การทดลองเพิ่มเติม (ไม่บังคับ):** เซลล์ถัดไปสั่งงานเดียวกันเป็น **ภาษาไทย** ให้รันเซลล์และเปรียบเทียบกับผลลัพธ์ภาษาอังกฤษ แบบจำลองขนาดเล็กมักตอบภาษาไทยได้ไม่คล่องหรือคลาดเคลื่อนมากกว่า ผู้เรียนสามารถแก้ไขข้อความภาษาไทยภายในเครื่องหมายอัญประกาศได้

In [ ]:
# 🌐 การทดลองเพิ่มเติม (ไม่บังคับ): สั่งงานเป็นภาษาไทย แล้วเปรียบเทียบกับผลลัพธ์ภาษาอังกฤษข้างต้น
thai_user = f"""กรุณาสรุปบันทึกทางคลินิกด้านล่างเป็นภาษาไทยจำนวน 3 หัวข้อ สำหรับแพทย์เวชปฏิบัติทั่วไป
โดยใช้เฉพาะข้อมูลที่ปรากฏในบันทึก และไม่เพิ่มเติมข้อมูลอื่น

บันทึก:
{note}"""
print(chat("You are a helpful clinical assistant. Answer in Thai.", thai_user))

✍️ **คำถามที่ 1:** จงเปรียบเทียบสรุปจากพรอมต์ที่ไม่ชัดเจนกับสรุปจากพรอมต์ที่ปรับปรุงแล้ว โปรดระบุความแตกต่าง 2 ประการ และระบุว่ามีสรุปใดที่เพิ่มข้อมูลซึ่ง **ไม่ปรากฏ** ในบันทึกหรือไม่ (หากได้ทดลองพรอมต์ภาษาไทย โปรดอธิบายความแตกต่างของผลลัพธ์โดยสังเขป)

✍️ **คำตอบ:**



## 3. ผลลัพธ์แบบมีโครงสร้าง (JSON)

ข้อความอิสระเหมาะสำหรับผู้อ่านที่เป็นมนุษย์ แต่ระบบเวชระเบียนอิเล็กทรอนิกส์ (electronic health record, EHR) ทะเบียนผู้ป่วย (registry) และแดชบอร์ด (dashboard) ต้องการ **ข้อมูลมีโครงสร้าง (structured data)** **JSON** เป็นรูปแบบข้อความมาตรฐานสำหรับจุดประสงค์นี้ ประกอบด้วยช่องข้อมูลที่มีชื่อกำกับภายในวงเล็บปีกกา เช่น `{"age": 57, "sex": "M", "medications": ["metformin"]}` ในส่วนนี้จะกำหนดให้แบบจำลองตอบเป็น JSON *เท่านั้น* แล้วใช้ฟังก์ชันช่วย `parse_json_reply()` แปลงคำตอบเป็น dictionary ของ Python ฟังก์ชันนี้รองรับข้อความส่วนเกิน เช่น เครื่องหมาย ```` ```json ```` ซึ่งแบบจำลองขนาดเล็กมักแทรกมาด้วย

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: การสกัดข้อมูลเป็นโครงสร้าง (JSON)
# เป้าหมายทางคลินิก: แปลงบันทึกข้อความอิสระเป็นระเบียนข้อมูลมีโครงสร้างที่ระบบคอมพิวเตอร์นำไปใช้ได้
# ความหมายของพรอมต์ (ภาษาอังกฤษ): "ให้ตอบเป็น JSON เท่านั้น โดยมีคีย์ age, sex, conditions, vitals, plan"
# ✏️ สิ่งที่ต้องทำ (แก้ไข 2 ตำแหน่ง):
#   1. พรอมต์ขาดคีย์ 1 รายการ ให้เพิ่ม  medications (list)  ในรายการคีย์ (ภายในเครื่องหมายอัญประกาศ)
#   2. ที่บรรทัด `data = ...` ให้แทนที่ ... ด้วย  parse_json_reply(raw)
# 💡 คำใบ้: parse_json_reply เป็นฟังก์ชันช่วยที่จัดเตรียมไว้แล้ว ทำหน้าที่ค้นหาบล็อก {...} ในคำตอบ
#    และแปลงเป็น dictionary

system = "You are a clinical information extractor. Respond with ONLY valid JSON, no prose."
user = f"""Extract structured data from this note as JSON with these keys:
age (integer), sex ("M" or "F"), conditions (list), vitals (object), plan (list)

Note:
{note}"""

raw = chat(system, user, max_new_tokens=300)
print("คำตอบของแบบจำลอง:\n", raw, "\n")

data = ...   # ✏️ แทนที่ ... ด้วย parse_json_reply(raw)

show_record(data)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
system = "You are a clinical information extractor. Respond with ONLY valid JSON, no prose."
user = f"""Extract structured data from this note as JSON with these keys:
age (integer), sex ("M" or "F"), conditions (list), medications (list), vitals (object), plan (list)

Note:
{note}"""

raw = chat(system, user, max_new_tokens=300)
print("คำตอบของแบบจำลอง:\n", raw, "\n")

data = parse_json_reply(raw)

show_record(data)
print("\nรายการยา (medications):", data.get("medications"))

💡 **คำอธิบายผลลัพธ์**
แบบจำลองแปลงบันทึกที่เขียนเป็นประโยคให้เป็นช่องข้อมูลที่มีชื่อกำกับ เมื่อ `data` เป็น dictionary แล้ว โปรแกรมสามารถนำไปใช้ต่อได้ทันที เช่น `data["medications"]` ให้รายการยา ซึ่งอาจส่งต่อไปยังระบบตรวจสอบอันตรกิริยาระหว่างยา (drug interaction) หรือทะเบียนผู้ป่วยเบาหวาน อย่างไรก็ตาม แบบจำลองยังอาจผิดพลาด จึงต้องตรวจสอบทุกช่องข้อมูล

✍️ **คำถามที่ 2:** จงตรวจสอบระเบียนที่สกัดได้เทียบกับบันทึกทีละบรรทัด โรค (conditions) และยา (medications) ถูกต้องและครบถ้วนหรือไม่ มีข้อมูลใดตกหล่น หรือถูก **สร้างขึ้นเอง** หรือไม่ (เช่น การวินิจฉัย hypertension ซึ่งบันทึกไม่ได้ระบุไว้)

✍️ **คำตอบ:**



## 4. การสร้างข้อมูลเท็จ (hallucination) และความปลอดภัย

**การสร้างข้อมูลเท็จ (hallucination)** คือกรณีที่ LLM ตอบอย่างมั่นใจในสิ่งที่ไม่เป็นความจริงหรือไม่ปรากฏในแหล่งข้อมูล เช่น การสร้างค่าผลตรวจทางห้องปฏิบัติการขึ้นเอง ซึ่งเป็นอันตรายในงานสุขภาพ มาตรการป้องกันเบื้องต้นคือการกำหนด **กฎการอ้างอิงแหล่งข้อมูล (grounding rule)** ในพรอมต์ ให้ตอบจากบันทึกเท่านั้น และหากไม่มีข้อมูล ให้ตอบว่า "Not documented" (ไม่มีการบันทึก)

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: การทดสอบการสร้างข้อมูลเท็จ
# เป้าหมายทางคลินิก: ตรวจสอบว่าแบบจำลองสร้างข้อมูลที่ไม่ปรากฏในบันทึกหรือไม่
# บันทึกนี้ไม่มีผลตรวจคอเลสเตอรอล คำตอบที่ปลอดภัยจึงมีเพียง "Not documented"
# ความหมายของ SAFETY_RULE: "หากไม่มีข้อมูลในบันทึก ให้ตอบว่า 'Not documented' เท่านั้น ห้ามคาดเดา"
# ✏️ สิ่งที่ต้องทำ:
#   1. รันเซลล์โดยไม่แก้ไข และอ่านคำตอบทั้งสองแบบ (ไม่มีกฎความปลอดภัย / มีกฎความปลอดภัย)
#   2. เปลี่ยน `question` (ภายในเครื่องหมายอัญประกาศ) เป็นข้อมูลอื่นที่ไม่ปรากฏในบันทึก แล้วรันใหม่ เช่น
#      "What is the patient's weight?" (น้ำหนัก)  "What was the eGFR?"  "Which allergies does the patient have?" (ประวัติการแพ้)
#   3. ไม่บังคับ: แก้ไข SAFETY_RULE ให้เข้มงวดขึ้น

question = "What was the patient's LDL cholesterol level?"      # ✏️ ทดลองคำถามอื่น

SAFETY_RULE = "If the information is not in the note, answer exactly 'Not documented'. Never guess."   # ✏️ ไม่บังคับ

answer_plain = chat("You are a clinical assistant.", f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
answer_safe  = chat("You are a clinical assistant. " + SAFETY_RULE,
                    f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
print("❓", question)
print("\n--- ไม่มีกฎความปลอดภัย ---\n" + answer_plain)
print("\n--- มีกฎความปลอดภัย ---\n" + answer_safe)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ความหมายของกฎ: ตอบจากบันทึกเท่านั้น หากไม่มีข้อมูลให้ตอบ 'Not documented' ห้ามคาดเดาหรือใช้ค่าปกติทั่วไป
SAFETY_RULE = ("Answer ONLY from the note. If the information is not in the note, answer exactly "
               "'Not documented'. Never guess or use typical values.")
for question in ["What was the patient's LDL cholesterol level?",
                 "What is the patient's weight?",
                 "What was the eGFR?"]:
    answer_plain = chat("You are a clinical assistant.", f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
    answer_safe  = chat("You are a clinical assistant. " + SAFETY_RULE,
                        f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
    print("❓", question)
    print("--- ไม่มีกฎความปลอดภัย ---\n" + answer_plain)
    print("--- มีกฎความปลอดภัย ---\n" + answer_safe + "\n")

💡 **คำอธิบายผลลัพธ์**
หากไม่มีกฎ แบบจำลองอาจคาดเดาค่าที่พบบ่อย หรือให้คำแนะนำทั่วไปแทนการระบุว่าไม่มีข้อมูล กฎการอ้างอิงแหล่งข้อมูลมักช่วยลดปัญหานี้ได้ แต่พรอมต์ไม่ใช่หลักประกัน บุคลากรทางการแพทย์ยังต้องตรวจสอบคำตอบของ LLM ทุกครั้ง

✍️ **คำถามที่ 3:** สำหรับแต่ละคำถามที่ทดลอง แบบจำลองสร้างข้อมูลเท็จ (สร้างค่าหรือข้อเท็จจริงขึ้นเอง) หรือไม่ ทั้งในกรณีที่มีและไม่มีกฎความปลอดภัย โปรดยกตัวอย่างคำตอบประกอบ 1 ตัวอย่าง

✍️ **คำตอบ:**



✍️ **คำถามที่ 4:** จงอธิบายใน 2 ประโยคว่า เหตุใดการ **อ้างอิงแหล่งข้อมูล (grounding)** ที่เชื่อถือได้ (ปฏิบัติการถัดไป: retrieval-augmented generation, RAG) จึงมีความสำคัญต่อการใช้งานทางคลินิก และโปรดระบุมาตรการป้องกัน 1 ประการที่ผู้เรียนจะกำหนดก่อนนำ LLM ไปใช้กับบันทึกจริงในหน่วยงาน

✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** โค้ดตั้งต้น (baseline) ด้านล่างรันได้ทันทีและสร้างไฟล์ส่งผลที่ถูกต้อง ผู้เรียนทุกคนจึงสามารถส่งผลได้

ให้รันโปรแกรมสกัดข้อมูลกับ **บันทึกทางคลินิกสังเคราะห์ 20 ฉบับ** ของรายวิชา แล้วส่งค่า `age`, `sex`, `conditions` และ `medications` ของแต่ละบันทึก **ตัวชี้วัด: ค่าเฉลี่ยของ age accuracy, sex accuracy, conditions F1 และ medications F1** (F1 พิจารณาทั้งรายการที่ตกหล่นและรายการเกินที่ไม่ถูกต้อง ระบบยอมรับคำพ้องความหมาย เช่น *HTN* / *hypertension* หรือ *T2DM* / *type 2 diabetes* และเปรียบเทียบชื่อยาโดยไม่พิจารณาขนาดยา)

ไฟล์ส่งผล: `lab08_submission.json` — JSON object ที่มีคีย์เป็นรหัสบันทึก เช่น `{"N01": {"age": 57, "sex": "M", "conditions": [...], "medications": [...]}}` หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่า

> **หลักความซื่อสัตย์ทางวิชาการ (honor code):** ผลการทำนายต้องได้จากพรอมต์ แบบจำลอง หรือโค้ดของผู้เรียนเท่านั้น ห้ามอ่านบันทึกชุดทดสอบแล้วพิมพ์คำตอบด้วยตนเอง งานนี้เป็นงานเดี่ยว

ขั้นที่ 1 — พรอมต์ตั้งต้น (baseline) ทดสอบกับบันทึกฝึกหัด 3 ฉบับที่ทราบคำตอบ (ไม่ใช่ชุดทดสอบ) พร้อมคะแนนโดยประมาณ

In [ ]:
# 🏆 โปรแกรมสกัดข้อมูลตั้งต้น และคะแนนโดยประมาณบนบันทึกฝึกหัด 3 ฉบับ
# ความหมายของพรอมต์: ตอบเป็น JSON เท่านั้น มีคีย์ age (จำนวนเต็ม), sex ('M'/'F'), conditions (รายการโรค), medications (รายการชื่อยา)
EXTRACT_SYSTEM = ("You are a clinical information extractor. Respond with ONLY valid JSON with keys "
                  "age (integer), sex ('M' or 'F'), conditions (list of diagnoses), medications (list of drug names).")

def extract_note(text, system, examples=None, max_new_tokens=200):
    """รันแบบจำลองกับบันทึก 1 ฉบับ; examples = (ไม่บังคับ) รายการตัวอย่าง few-shot คู่ (ข้อความบันทึก, คำตอบแบบ dict)"""
    shots = [(f"Note:\n{n}", json.dumps(a)) for n, a in (examples or [])]
    raw = chat(system, f"Note:\n{text}", max_new_tokens=max_new_tokens, examples=shots)
    return clean_record(parse_json_reply(raw, quiet=True))

DEV_NOTES = [
    {"note": "72M with HTN, COPD and atrial fibrillation presents for follow-up. Reports ankle swelling x2 weeks. "
             "BP 146/88, HR 78. Medications: amlodipine 5mg daily; apixaban 5 mg BID; tiotropium inhaler daily. "
             "No fever. Plan: recheck BP in 4 weeks.",
     "gold": {"age": 72, "sex": "M", "conditions": ["hypertension", "copd", "atrial fibrillation"],
              "medications": ["amlodipine", "apixaban", "tiotropium"]}},
    {"note": "Patient is a 39 year old female with asthma and hypothyroidism, seen today for review. Reports night-time "
             "wheeze x1 week. BP 118/70, HR 84. Current meds include levothyroxine 75 mcg daily and montelukast 10 mg "
             "at night. Denies chest pain. Plan: review inhaler technique, TSH in 6 weeks.",
     "gold": {"age": 39, "sex": "F", "conditions": ["asthma", "hypothyroidism"],
              "medications": ["levothyroxine", "montelukast"]}},
    {"note": "61-year-old man with a history of CKD stage 3 and type 2 diabetes presents for medication review. Reports "
             "fatigue x1 month. BP 138/84, HR 72. HbA1c 7.9%. Medications: metformin 500mg BID; empagliflozin 10 mg "
             "daily; atorvastatin 40mg at bedtime. No chest pain. Plan: continue current regimen, recheck eGFR in 3 months.",
     "gold": {"age": 61, "sex": "M", "conditions": ["chronic kidney disease", "type 2 diabetes"],
              "medications": ["metformin", "empagliflozin", "atorvastatin"]}},
]

_ALIASES = {"htn": "hypertension", "high blood pressure": "hypertension", "af": "atrial fibrillation",
            "afib": "atrial fibrillation", "chronic obstructive pulmonary disease": "copd",
            "ckd": "chronic kidney disease", "t2dm": "type 2 diabetes", "dm2": "type 2 diabetes",
            "type 2 diabetes mellitus": "type 2 diabetes", "diabetes mellitus type 2": "type 2 diabetes"}

def _norm_cond(c):
    c = re.sub(r"\s*\(.*?\)|\bstage \w+\b", "", str(c).lower()).strip(" .")
    return _ALIASES.get(c, c)

def _norm_med(m):
    words = str(m).lower().split()
    return words[0] if words else ""

def _f1(pairs):
    tp = sum(len(set(g) & set(p)) for g, p in pairs)
    n_pred, n_gold = sum(len(set(p)) for _, p in pairs), sum(len(set(g)) for g, _ in pairs)
    return 2 * tp / (n_pred + n_gold) if n_pred + n_gold else 1.0

def rough_score(system, examples=None, show=True):
    """ประมาณค่าตัวชี้วัดของตารางอันดับบน DEV_NOTES (ระบบให้คะแนนจริงรองรับคำพ้องความหมายมากกว่านี้)"""
    preds = [extract_note(d["note"], system, examples) for d in DEV_NOTES]
    golds = [d["gold"] for d in DEV_NOTES]
    if show:
        for p, g in zip(preds, golds):
            print("ผลการทำนาย:", p, "\nเฉลย      :", g, "\n")
    age = sum(p["age"] == g["age"] for p, g in zip(preds, golds)) / len(golds)
    sex = sum(p["sex"] == g["sex"] for p, g in zip(preds, golds)) / len(golds)
    cond = _f1([([_norm_cond(c) for c in g["conditions"]], [_norm_cond(c) for c in p["conditions"]]) for p, g in zip(preds, golds)])
    med = _f1([([_norm_med(m) for m in g["medications"]], [_norm_med(m) for m in p["medications"]]) for p, g in zip(preds, golds)])
    print(f"age acc {age:.2f} | sex acc {sex:.2f} | conditions F1 {cond:.2f} | medications F1 {med:.2f} "
          f"| คะแนนโดยประมาณ {(age + sex + cond + med) / 4:.3f}")
    return (age + sex + cond + med) / 4

baseline_score = rough_score(EXTRACT_SYSTEM)

**แนวทางการปรับปรุงให้ได้คะแนนสูงกว่า baseline**

- ระบุโครงสร้าง JSON (schema) ในพรอมต์อย่างชัดเจน พร้อมตัวอย่างรูปแบบผลลัพธ์ที่ต้องการ
- เพิ่ม **ตัวอย่างแบบ few-shot** 1–2 ตัวอย่าง (บันทึกพร้อมคำตอบที่ถูกต้อง) ดังที่แสดงใน `MY_EXAMPLES` ด้านล่าง
- กำหนดขอบเขตของโรคให้ชัดเจน: นับเฉพาะการวินิจฉัยจากประวัติ ("with ...", "history of ...") **ไม่นับ** อาการนำ ("Reports knee pain") หรือสิ่งที่ผู้ป่วยปฏิเสธ ("No chest pain", "Denies dyspnea")
- กำหนดว่า "copy diagnoses exactly as written" (ระบบให้คะแนนยอมรับคำพ้องความหมายและคำย่อ) และ "drug names only, without dose or frequency"
- กำหนด `max_new_tokens` ให้เหมาะสม และใช้ Python ประมวลผลต่อ (เช่น ตัดรายการซ้ำ หรือใช้ regex ระบุอายุและเพศ)
- ใช้แบบจำลองแบบเปิดที่มีขนาดใหญ่ขึ้น เช่น `Qwen2.5-3B-Instruct` ซึ่งทำงานบน T4 แบบ fp16 ได้ (เปลี่ยนค่า `MODEL` แล้วโหลดใหม่)

ขั้นที่ 2 — พรอมต์ที่ผู้เรียนปรับปรุง (ไม่บังคับ; แก้ไขข้อความแล้วรันใหม่เพื่อเปรียบเทียบกับคะแนน baseline)

In [ ]:
# 🏆 ไม่บังคับ — พรอมต์ที่ผู้เรียนปรับปรุง: แก้ไขข้อความภายในเครื่องหมายอัญประกาศ รันใหม่ แล้วเปรียบเทียบคะแนนโดยประมาณ
# ความหมายของข้อความที่เพิ่ม: conditions = โรคที่ผู้ป่วยเป็น (จากประวัติ) คัดลอกตามที่บันทึก ไม่รวมอาการนำหรือสิ่งที่ปฏิเสธ;
# medications = ชื่อยาเท่านั้น ไม่รวมขนาดยาและความถี่
MY_SYSTEM = EXTRACT_SYSTEM + (
    " Conditions = diagnoses the patient HAS (from the history), copied as written; do NOT include presenting"
    " symptoms or negated findings such as 'No chest pain'. Medications = drug names only, without dose or frequency.")

MY_EXAMPLES = [   # ตัวอย่างแบบ few-shot: (ข้อความบันทึก, คำตอบที่ถูกต้อง)
    (note, {"age": 57, "sex": "M", "conditions": ["T2DM"], "medications": ["metformin"]}),
]

my_score = rough_score(MY_SYSTEM, MY_EXAMPLES)
print(f"\nbaseline {baseline_score:.3f}  →  พรอมต์ที่ปรับปรุง {my_score:.3f}")

ขั้นที่ 3 — รันกับบันทึก 20 ฉบับของรายวิชา บันทึกไฟล์ `lab08_submission.json` และส่งผล (ใช้เวลาประมาณ 2–4 นาทีบน GPU T4)

In [ ]:
# 🏆 สกัดข้อมูลจากบันทึกชุดทดสอบทั้ง 20 ฉบับ และส่งผล
FINAL_SYSTEM, FINAL_EXAMPLES = EXTRACT_SYSTEM, None   # ✏️ เปลี่ยนเป็น MY_SYSTEM, MY_EXAMPLES หากพรอมต์ที่ปรับปรุงได้คะแนนสูงกว่า

notes = json.load(open(lab_data("notes/notes.json")))
submission = {}
for i, n in enumerate(notes, 1):
    submission[n["id"]] = extract_note(n["note"], FINAL_SYSTEM, FINAL_EXAMPLES)
    print(f"{i:2d}/{len(notes)} {n['id']}: {submission[n['id']]}")

json.dump(submission, open("lab08_submission.json", "w"), indent=1)
print("\nบันทึกไฟล์ lab08_submission.json เรียบร้อย จำนวน", len(submission), "บันทึก")
if NAME and JOIN_CODE:
    submit_to_leaderboard("lab08_submission.json")
else:
    print("หากต้องการส่งผล โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า รันเซลล์ดังกล่าว แล้วรันเซลล์นี้อีกครั้ง")

## ชิ้นงานที่ต้องส่ง

ปฏิบัติการนี้เป็น **งานเดี่ยว** ให้ส่งชิ้นงานตามช่องทางที่รายวิชากำหนด ดังนี้

1. โน้ตบุ๊กนี้ซึ่ง **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (แสดงผลลัพธ์ของสรุปที่ปรับปรุงแล้ว การสกัดข้อมูล JSON ที่แปลงได้สำเร็จ และการทดสอบการสร้างข้อมูลเท็จ)
2. คำตอบ ✍️ ของคำถามที่ 1–4 (รวมถึงข้อสรุป 2 ประโยคเกี่ยวกับความสำคัญของการอ้างอิงแหล่งข้อมูล)

*ไม่บังคับ:* ผู้เรียนที่เข้าร่วมโจทย์ท้าทายบน 🏆 ตารางอันดับ สามารถระบุชื่อที่ใช้และคะแนนสูงสุดไว้ในส่วนนี้ ทั้งนี้ไม่ใช่ข้อกำหนดของชิ้นงาน